In [ ]:
import chardet
import json
import pandas as pd
from collections.abc import Mapping
from collections import OrderedDict
from typing import Any, List
from pathlib import Path

### CAUSALITY Project Noteboook

A notebook to ingest vuln data, add rating labels, and calculate stats on ratings and watchlist coverage. 
Resale and /or incorporation into a paid product or service is not covered by license.   
See https://github.com/opendr-io/causality/blob/main/LICENSE.md for details.  

Usage: In a venv, install any missing modules in the cell above. Next define the input files below. The next cell will do a directory listing of the current path to help locate where the notebook is relative to the data files.

vuln_path - your vuln data in csv format. Most of the export I see are in csv, let me know if you need a json ingestor.  
kev_path - the name of the kev data file in csv format.  
epss_path - (optional) the name of the epss score data file in csv format.  
file_path4 - the name of your ratings file for 2024 (from the causality project; start with this one: https://github.com/opendr-io/causality/blob/main/2024/reduction.txt)  
file_path5 - the name of your ratings file for 2025 (from the causality project; start with this one: https://github.com/opendr-io/causality/blob/main/2024/reduction.txt)  

In [ ]:
print("Current directory is:")
print(Path.cwd())
print("Here is the current directory listing - make sure your data files are here:")
for p in sorted(Path(".").iterdir(), key=lambda x: (not x.is_dir(), x.name.lower())):
    tag = "<DIR>" if p.is_dir() else "     "
    print(f"{tag}  {p.name}")

In [ ]:
# Define your file names here
vuln_path = 'nvd_export.csv' # Identify the vuln data file to ingest
kev_path = 'kev.csv' # Download from the KEV
epss_path = 'epss.csv' # Optionally ingest epsss scores (download from the epss project)
# Provide the ratings files from the causality project https://github.com/opendr-io/causality
file_path4 = '2024.txt'
file_path5 = 'august-2025-combined-ratings.txt'

In [ ]:
# Load 2025 rating scores 
with open(file_path5, 'rb') as file:
    result = chardet.detect(file.read())
    encoding = result['encoding']

rating2025 = pd.read_csv(file_path5, low_memory=False, encoding=encoding)
print("Unique value counts per field:\n")
for col in rating2025.columns:
    n_unique = rating2025[col].nunique(dropna=True)
    print(f"{col}: {n_unique} unique values")

all_nan = [col for col in rating2025.columns if rating2025[col].isna().all()]
if all_nan:
    print("\n⚠️ Fields entirely NaN:")
    for col in all_nan:
        print(f" - {col}")
else:
    print("\n✅ No fields are entirely NaN")

print("Shape of the 2025 ratings dataframe is:", rating2025.shape)

In [ ]:
# Load 2024 rating scores
with open(file_path4, 'rb') as file:
    result = chardet.detect(file.read())
    encoding = result['encoding']
rating2024 = pd.read_csv(file_path4, sep='\t', low_memory=False, encoding=encoding)

print("Unique value counts per field:\n")
for col in rating2024.columns:
    n_unique = rating2024[col].nunique(dropna=True)
    print(f"{col}: {n_unique} unique values")

all_nan = [col for col in rating2024.columns if rating2024[col].isna().all()]
if all_nan:
    print("\n⚠️ Fields entirely NaN:")
    for col in all_nan:
        print(f" - {col}")
else:
    print("\n✅ No fields are entirely NaN")

print("Shape of the 2024 ratings dataframe is:", rating2024.shape)

In [ ]:
# Load EPSS scores (skip the first descriptive line)
df_epss = pd.read_csv(epss_path, skiprows=1)
print("Unique value counts per field:\n")
for col in df_epss.columns:
    n_unique = df_epss[col].nunique(dropna=True)
    print(f"{col}: {n_unique} unique values")

# Identify fields that are entirely NaN
all_nan = [col for col in df_epss.columns if df_epss[col].isna().all()]

if all_nan:
    print("\n⚠️ Fields entirely NaN:")
    for col in all_nan:
        print(f" - {col}")
else:
    print("\n✅ No fields are entirely NaN")

print("Shape of the epss dataframe is:", df_epss.shape)

In [ ]:
# Read KEV into a DataFrame
kev = pd.read_csv(kev_path)

all_nan = [col for col in kev.columns if kev[col].isna().all()]
if all_nan:
    print("\n⚠️ Fields entirely NaN:")
    for col in all_nan:
        print(f" - {col}")
else:
    print("\n✅ No fields are entirely NaN")

print("Shape of the kev dataframe is:", kev.shape)
kev = kev.rename(columns=str.lower)
kev = kev.rename(columns={'cveid': 'cve'})
print(kev.columns.tolist())

# Extract year from 'cve' like 'cve-2023-6129' -> 2023
kev['year'] = (
    kev['cve']
      .astype('string')
      .str.extract(r'(?i)cve[-_]?(\d{4})', expand=False)
      .astype('Int64')   # pandas nullable integer; keeps NaN for non-matches
)
has_year = kev['year'].notna()
with_year = int(has_year.sum())
without_year = int((~has_year).sum())
total = len(kev)

print(f"With year: {with_year:,}")
print(f"Without year: {without_year:,}")
print(f"Total: {total:,}")
print(f"Coverage: {with_year/total:.2%}")

In [ ]:

with open(vuln_path, 'rb') as file:
    result = chardet.detect(file.read())
    encoding = result['encoding']
vulns = pd.read_csv(vuln_path, low_memory=False, encoding=encoding, header=0)
vulns.columns = vulns.columns.str.strip().str.lower()

all_nan = [col for col in vulns.columns if vulns[col].isna().all()]
if all_nan:
    print("\n⚠️ Fields entirely NaN:")
    for col in all_nan:
        print(f" - {col}")
else:
    print("\n✅ No fields are entirely NaN")

print("Shape of the vulns dataframe is:", vulns.shape)
print()
print(vulns.columns.tolist())

In [ ]:
# Check out the field list above and identify your field that contains CVE IDs. Provide it to the function below
# so that we have normalized field names across dataframes.

SOURCE_CVE_FIELD = 'FIELD'  # <-- change this as needed
colmap = {c.lower(): c for c in vulns.columns}

if SOURCE_CVE_FIELD.lower() in colmap:
    src = colmap[SOURCE_CVE_FIELD.lower()]
    if src == 'cve':
        pass  # already named 'cve'
    elif 'cve' in vulns.columns:
        print("Target column 'cve' already exists; skipping rename to avoid duplicate.")
    else:
        vulns.rename(columns={src: 'cve'}, inplace=True)
else:
    print(f"Column '{SOURCE_CVE_FIELD}' not found; nothing to rename.")


In [ ]:
# Extract year from 'cve' (e.g., 'CVE-2023-6129' -> 2023) into a nullable Int column
if 'cve' not in vulns.columns:
    raise KeyError("Expected a 'cve' column in the 'vulns' DataFrame.")

vulns['year'] = (
    vulns['cve']
        .astype('string')
        .str.extract(r'(?i)cve[-_]?(\d{4})', expand=False)
        .astype('Int64')
)

# Report counts
has_year = vulns['year'].notna()
with_year = int(has_year.sum())
without_year = int((~has_year).sum())
total = len(vulns)

print(f"With year: {with_year:,}")
print(f"Without year: {without_year:,}")
print(f"Total: {total:,}")
print(f"Coverage: {with_year/total:.2%}" if total else "Coverage: N/A")


In [ ]:
# Add a boolean 'kev' column to vulns indicating if its CVE appears in kev

if 'cve' not in vulns.columns:
    raise KeyError("vulns is missing 'cve' column")
if 'cve' not in kev.columns:
    raise KeyError("kev is missing 'cve' column")

kev_lookup = set(
    kev['cve'].astype('string').str.strip().str.lower().dropna().unique()
)
vulns_cve_norm = vulns['cve'].astype('string').str.strip().str.lower()
vulns['kev'] = vulns_cve_norm.isin(kev_lookup).fillna(False)
vulns['kev'] = vulns['kev'].astype(bool)

In [ ]:
total = len(vulns)
matches = int(vulns['kev'].sum())                  # count of True
populated = int(vulns['kev'].notna().sum())        # rows where 'kev' is not NaN
non_matches = populated - matches                  # or: int((~vulns['kev']).sum())

print(f"KEV matches (True): {matches:,}")
print(f"Non-matches (False): {non_matches:,}")
print(f"Rows populated in 'kev': {populated:,} (out of {total:,})")

In [ ]:
# Make sure cve fields are strings
vulns["cve"] = vulns["cve"].astype(str)
df_epss["cve"] = df_epss["cve"].astype(str)

vulns = vulns.merge(df_epss, on="cve", how="left", suffixes=("", "_epss"))
vulns["in_epss"] = vulns["epss"].notna()
total = len(vulns)
matches = vulns["in_epss"].sum()
missing = total - matches

print(f"✅ {matches} CVEs matched with EPSS")
if missing > 0:
    print(f"⚠️ {missing} CVEs in NVD had no EPSS match")
import pandas as pd

# --- Validate required columns ---
required = ['in_epss', 'cve']
missing_required = [c for c in required if c not in vulns.columns]
if missing_required:
    raise KeyError(f"Missing required column(s) in 'vulns': {missing_required}")

# Optional grouping columns
group_fields = []
missing_opts = []
if 'cvssdata.baseseverity' in vulns.columns:
    group_fields.append('cvssdata.baseseverity')
else:
    missing_opts.append('cvssdata.baseseverity')

if 'vulnstatus' in vulns.columns:
    group_fields.append('vulnstatus')
else:
    missing_opts.append('vulnstatus')

# Filter rows not in EPSS 
missing_epss = vulns[vulns['in_epss'].astype('boolean').fillna(False) == False].copy()

# Extract year from CVE (CVE-YYYY-XXXX)
missing_epss['year'] = (
    missing_epss['cve']
        .astype('string')
        .str.extract(r'(?i)cve[-_]?(\d{4})', expand=False)
        .astype('Int64')
)

# Group & summarize 
group_cols = ['year'] + group_fields if group_fields else ['year']
summary = (
    missing_epss
    .groupby(group_cols, dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values(['year', 'count'], ascending=[True, False])
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 0)
pd.set_option("display.max_colwidth", None)

if missing_opts:
    print(f"⚠️ Skipping missing columns: {', '.join(missing_opts)}")
print("⚠️ CVEs missing EPSS grouped by available fields:\n")
summary


In [ ]:


# Ensure required fields exist
for col in ['year', 'kev', 'in_epss']:
    if col not in vulns.columns:
        raise KeyError(f"'vulns' is missing required column: {col}")

# Work on a copy & normalize booleans (accepts True/False/Yes/No/1/0)
df = vulns[['year', 'kev', 'in_epss']].copy()

for col in ['kev', 'in_epss']:
    s = df[col]
    # keep real booleans; coerce common string/number truthy values
    df[col] = (
        s.where(s.isin([True, False]))
         .astype('string')
         .str.strip()
         .str.lower()
         .isin(['true', 't', '1', 'yes', 'y'])
    )

# Group summary: year x kev x in_epss
summary = (
    df.groupby(['year', 'kev', 'in_epss'], dropna=False)
      .size()
      .reset_index(name='count')
      .sort_values(['year', 'count'], ascending=[True, False])
)

# Add within-year totals and percentages
summary['year_total'] = summary.groupby('year')['count'].transform('sum')
summary['pct_of_year'] = (summary['count'] / summary['year_total']).round(4)

print("Summary by year × KEV × in_EPSS:")
print(summary.to_string(index=False))

# Optional: pivot for a compact table of counts
pivot = summary.pivot_table(
    index='year', columns=['kev', 'in_epss'],
    values='count', aggfunc='sum', fill_value=0
)
print("Using the year counts below, you can decide to use 2024 or 2025 rating data:")

# If you want to exclude rows where year is NaN, add:
# df = df[df['year'].notna()].copy()


In [ ]:

for col in ['cve', 'year']:
    if col not in vulns.columns:
        raise KeyError(f"'vulns' is missing required column: {col}")
if 'rating' not in vulns.columns:
    vulns['rating'] = 'cold'
else:
    vulns['rating'] = vulns['rating'].fillna('cold')

try:
    r25_raw = ratings2025
except NameError:
    try:
        r25_raw = rating2025
    except NameError:
        raise NameError("Provide a dataframe named 'ratings2025' (or 'rating2025') with CVE ratings for 2025.")

# Normalize columns and build CVE->rating map
r25 = r25_raw.copy()
r25.columns = r25.columns.str.strip().str.lower()
cve_col_r25 = next((c for c in ['cve', 'cveid', 'cve_id'] if c in r25.columns), None)
if cve_col_r25 is None:
    raise KeyError("ratings2025 is missing a CVE column (expected 'cve', 'cveid', or 'cve_id').")
if 'rating' not in r25.columns:
    raise KeyError("ratings2025 is missing required 'rating' column.")

r25['_cve_norm'] = r25[cve_col_r25].astype('string').str.strip().str.lower()
cve_to_rating = (
    r25.dropna(subset=['rating'])
       .drop_duplicates(subset=['_cve_norm'])
       .set_index('_cve_norm')['rating']
)

# Update 'rating' for 2025 rows using mapping; keep 'cold' where no match
mask_2025 = vulns['year'].astype('Int64') == 2025
mapped = (
    vulns.loc[mask_2025, 'cve']
         .astype('string').str.strip().str.lower()
         .map(cve_to_rating)
)
vulns.loc[mask_2025, 'rating'] = mapped.fillna(vulns.loc[mask_2025, 'rating'])


print(f"Ratings mapped for 2025 rows: {int(mapped.notna().sum())}/{int(mask_2025.sum())}")
if 'rating' in vulns.columns:
    rating_col = 'rating'
elif 'rating' not in vulns.columns:
    raise KeyError("Neither 'rating' nor 'ratings' column found in 'vulns'.")
    
counts = vulns[rating_col].value_counts(dropna=False)
perc = (counts / len(vulns)).mul(100).round(2)
print(f"Counts for '{rating_col}':")
print(counts.to_string())
print("\nPercentages:")
print(perc.astype(str) + '%')
print("\nOrdered (hot, warm, cold):")
print(vulns[rating_col].value_counts().reindex(['hot','warm','cold'], fill_value=0))

In [ ]:
# Ensure required columns exist
for col in ['kev', 'rating']:
    if col not in vulns.columns:
        raise KeyError(f"'vulns' is missing required column: {col}")

# Count unique (kev, rating) combinations
combo_counts = (
    vulns.groupby(['kev', 'rating'], dropna=False)
         .size()
         .reset_index(name='count')
         .sort_values('count', ascending=False)
)

# Add percentages
combo_counts['percent'] = (combo_counts['count'] / len(vulns) * 100).round(2)

print("Unique combinations of KEV × Rating:")

# Optional: a compact pivot view (ratings as rows, KEV as columns)
pivot = combo_counts.pivot_table(index='rating', columns='kev', values='count', fill_value=0)
print("\nPivot (counts):")
pivot

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)
pd.set_option("display.width", 0)
pd.set_option("display.max_colwidth", 100)
combo_counts

In [ ]:
# Ensure required columns exist
for col in ['kev', 'rating']:
    if col not in vulns.columns:
        raise KeyError(f"'vulns' is missing required column: {col}")

# Normalize fields
rating_norm = vulns['rating'].astype('string').str.strip().str.lower()
kev_true = vulns['kev'].astype('string').str.strip().str.lower().isin(['true','t','1','yes','y'])

# Overall totals & percentages by rating (all rows)
total_all = len(vulns)
overall = (
    rating_norm.value_counts(dropna=False)
    .rename_axis('rating')
    .reset_index(name='count_all')
    .sort_values('count_all', ascending=False)
    .reset_index(drop=True)
)
overall['pct_all'] = (overall['count_all'] / total_all * 100).round(2)

# Within KEV=True: counts and % within KEV=True universe
subset = rating_norm[kev_true]
total_kev = len(subset)
by_kev = (
    subset.value_counts(dropna=False)
    .rename_axis('rating')
    .reset_index(name='count_in_kev_true')
    .sort_values('count_in_kev_true', ascending=False)
    .reset_index(drop=True)
)
by_kev['pct_within_kev_true'] = (
    (by_kev['count_in_kev_true'] / total_kev * 100).round(2) if total_kev else 0
)

# Merge into one nicely formatted table
summary = overall.merge(by_kev, on='rating', how='outer').fillna(0)
summary[['count_all','count_in_kev_true']] = summary[['count_all','count_in_kev_true']].astype(int)

# Optional: order common ratings first
order = pd.Categorical(summary['rating'], categories=['hot','warm','cold'], ordered=True)
summary = (
    summary.assign(_order=order)
           .sort_values(['_order','count_all'], ascending=[True, False])
           .drop(columns=['_order'])
           .reset_index(drop=True)
)

print("Totals and percentages by rating (overall and within KEV=True):")
summary